In [ ]:
# MLB 투수 부상 예측 - 1) XGBoost + Bayesian Optimization
# 불펜/선발 두 역할, 이진분류(부상 O/X)와 3종분류(어깨/팔꿈치 구분) 모두 학습/평가한다.
# 데이터: data/bullpen_dataset.csv, data/starter_dataset.csv (컬럼 설명은 data/data_description.md 참고)

import pandas as pd
import xgboost as xgb
from bayes_opt import BayesianOptimization
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
# 데이터 로드. label: 0=안다침, 1=어깨, 2=팔꿈치, 3=그 외 부상. split으로 train/val/test 구분됨

DATA = {
    "bullpen": pd.read_csv("data/bullpen_dataset.csv"),
    "starter": pd.read_csv("data/starter_dataset.csv"),
}

ID_COLS = ["player_id", "window_end_date", "il_start_date", "injury_class_strict", "days_to_injury", "split"]
CATEGORICAL_COLS = ["p_throws", "birth_country"]


def load_role(role, exclude_other=True, binarize=False):
    """역할(bullpen/starter)의 데이터를 train/val/test로 나눠서 반환.
    exclude_other=True면 label==3('그 외' 부상) 행 제외.
    binarize=True면 1,2를 합쳐서 0(안다침) vs 1(어깨 또는 팔꿈치) 이진분류로 변환."""
    df = DATA[role].copy()
    if exclude_other:
        df = df[df["label"] != 3].copy()
    if binarize:
        df["label"] = (df["label"] > 0).astype(int)
    for c in CATEGORICAL_COLS:
        df[c] = df[c].astype("category")
    return {s: df[df["split"] == s].reset_index(drop=True) for s in ("train", "val", "test")}


def numeric_feature_cols(df):
    return [c for c in df.columns if c not in ID_COLS + CATEGORICAL_COLS + ["label"]]

In [ ]:
# 입력변수 정리: 구종 그룹별 지표들끼리 상관관계가 매우 높은 경우가 많아(예: 전체
# 평균 구속과 직구 평균 구속) 상관계수 0.9 초과 컬럼은 미리 제거한다.
# train 데이터 기준으로만 계산해서 val/test로 정보가 새어들어가는 걸 막는다.

def select_uncorrelated_features(train_df, candidate_cols, threshold=0.9):
    corr = train_df[candidate_cols].corr().abs()
    kept = []
    for col in candidate_cols:
        is_redundant = any(pd.notna(corr.loc[col, k]) and corr.loc[col, k] > threshold for k in kept)
        if not is_redundant:
            kept.append(col)
    print(f"[feature_selection] 후보 {len(candidate_cols)}개 -> 상관관계(>|{threshold}|) 제거 후 {len(kept)}개")
    return kept

In [ ]:
# XGBoost 하이퍼파라미터 탐색 범위 (Bayesian Optimization으로 6개 탐색)

CORR_THRESHOLD = 0.9
PBOUNDS = {
    "max_depth": (3, 10),          # 트리 깊이
    "learning_rate": (0.01, 0.3),  # 각 트리 반영 비율
    "subsample": (0.5, 1.0),       # 트리마다 샘플링할 행 비율
    "colsample_bytree": (0.5, 1.0),  # 트리마다 샘플링할 컬럼 비율
    "min_child_weight": (1, 7),    # 리프 노드 최소 샘플 가중치 합
    "reg_lambda": (0.5, 5.0),      # L2 정규화 강도
}


def xgb_base_params(label_mode, max_depth, learning_rate, subsample, colsample_bytree,
                     min_child_weight, reg_lambda):
    common = dict(
        max_depth=int(round(max_depth)), learning_rate=learning_rate, n_estimators=2000,
        subsample=subsample, colsample_bytree=colsample_bytree, min_child_weight=min_child_weight,
        reg_lambda=reg_lambda, tree_method="hist", enable_categorical=True,
        early_stopping_rounds=50, random_state=42,
    )
    if label_mode == "binary":
        return dict(objective="binary:logistic", eval_metric="logloss", **common)
    return dict(objective="multi:softprob", num_class=3, eval_metric="mlogloss", **common)


def auc_score(y, proba, label_mode):
    if label_mode == "binary":
        return roc_auc_score(y, proba[:, 1])
    return roc_auc_score(y, proba, average="macro", multi_class="ovr")

In [ ]:
# 학습 + 평가 함수. 클래스 불균형(양성 표본 1~2%뿐)을 보정하기 위해 class-balanced
# sample weight를 train에 적용한다(val/test는 실제 분포 그대로 평가).

def evaluate(model, X, y, name, label_mode):
    proba = model.predict_proba(X)
    pred = model.predict(X)
    print(f"\n--- {name} (n={len(y):,}) ---")
    print(classification_report(y, pred, digits=3, zero_division=0))
    print("confusion matrix (행=실제, 열=예측):")
    print(confusion_matrix(y, pred))
    auc = auc_score(y, proba, label_mode)
    print(f"AUC: {auc:.3f}")
    return auc


def run(role, label_mode, n_iter=25, init_points=8):
    print(f"\n{'=' * 70}\n[XGBoost:{label_mode}] {role.upper()}\n{'=' * 70}")
    splits = load_role(role, exclude_other=True, binarize=(label_mode == "binary"))
    all_num_cols = numeric_feature_cols(splits["train"])
    kept_num_cols = select_uncorrelated_features(splits["train"], all_num_cols, threshold=CORR_THRESHOLD)
    feature_cols = kept_num_cols + CATEGORICAL_COLS

    xy = {s: (df[feature_cols], df["label"].astype(int)) for s, df in splits.items()}
    print(f"train={len(xy['train'][0]):,} val={len(xy['val'][0]):,} test={len(xy['test'][0]):,}")

    X_train, y_train = xy["train"]
    X_val, y_val = xy["val"]
    sample_weight = compute_sample_weight("balanced", y_train)

    def objective(max_depth, learning_rate, subsample, colsample_bytree, min_child_weight, reg_lambda):
        params = xgb_base_params(label_mode, max_depth, learning_rate, subsample,
                                  colsample_bytree, min_child_weight, reg_lambda)
        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train, sample_weight=sample_weight, eval_set=[(X_val, y_val)], verbose=False)
        try:
            return auc_score(y_val, model.predict_proba(X_val), label_mode)
        except ValueError:
            return 0.0

    optimizer = BayesianOptimization(f=objective, pbounds=PBOUNDS, random_state=42, verbose=0)
    optimizer.maximize(init_points=init_points, n_iter=n_iter)
    best = optimizer.max
    print(f"최적 하이퍼파라미터: {best['params']}  (탐색 중 최고 val AUC={best['target']:.4f})")

    p = best["params"]
    final_params = xgb_base_params(label_mode, p["max_depth"], p["learning_rate"], p["subsample"],
                                    p["colsample_bytree"], p["min_child_weight"], p["reg_lambda"])
    X_test, y_test = xy["test"]

    model = xgb.XGBClassifier(**final_params)
    model.fit(X_train, y_train, sample_weight=sample_weight, eval_set=[(X_val, y_val)], verbose=False)

    val_auc = evaluate(model, X_val, y_val, "Validation (최적 하이퍼파라미터)", label_mode)
    test_auc = evaluate(model, X_test, y_test, "Test (최적 하이퍼파라미터)", label_mode)
    return {
        "role": role, "label_mode": label_mode, "n_features": len(feature_cols),
        "best_params": p, "val_auc": val_auc, "test_auc": test_auc,
    }

In [ ]:
# 실행: 불펜/선발 x 이진분류/3종분류 총 4가지 조합을 전부 학습한다.
# 시간이 오래 걸리면 ROLES/LABEL_MODES를 줄여서 원하는 조합만 실행해도 된다.

ROLES = ["bullpen", "starter"]
LABEL_MODES = ["binary", "3class"]
N_ITER = 25        # 베이지안 탐색 반복 횟수
INIT_POINTS = 8    # 초기 무작위 탐색 횟수

results = []
for role in ROLES:
    for label_mode in LABEL_MODES:
        results.append(run(role, label_mode, N_ITER, INIT_POINTS))

In [ ]:
# 결과 요약

results_df = pd.DataFrame(results)[["role", "label_mode", "n_features", "val_auc", "test_auc"]]
results_df